# 📊 LinkedIn Jobs Data Explorer

MongoDB 및 덤프 파일(`data/silver_jobs.gz`)에 적재된 채용 공고 데이터를 살펴봅니다.

### 1. MongoDB 연결 라이브러리 및 Pandas 로드

In [ ]:
import os
import pandas as pd
from pymongo import MongoClient

### 2. DB 연결 및 덤프 안내

In [ ]:
dump_path = "../data/silver_jobs.gz"
if os.path.exists(dump_path):
    print(f"✅ 로컬 덤프 파일 확인됨: {dump_path}")
    print("💡 복원이 필요할 시: 'mongorestore --gzip --archive=data/silver_jobs.gz' 명령을 수행해 주세요.")

# Docker 네트워크 내부 호스트명 'mongodb'로 접근
try:
    client = MongoClient("mongodb://mongodb:27017")
    db = client.linkedin
    print("Collections:", db.list_collection_names())
except Exception as e:
    print("❌ MongoDB에 접근할 수 없습니다.", e)

### 3. Silver Layer (실시간 정제 데이터) 로드 및 조회

In [ ]:
try:
    jobs_coll = db['silver.jobs']
    df = pd.DataFrame(list(jobs_coll.find().limit(1000)))

    if not df.empty:
        if '_id' in df.columns:
            df = df.drop(columns=['_id'])
        print(f"📊 로드 완료: 총 {len(df)}개 공고")
        display(df.head())
    else:
        print("📭 데이터가 비어있습니다. 수집 작업을 먼저 진행하거나 덤프를 복원해 주세요.")
except Exception as e:
    print("데이터 로드 실패:", e)